# 作业2b：岭回归（加了L2正则化）

岭回归就是在普通线性回归的基础上多了一个L2正则化项，
让权重不要太大，防止过拟合。

损失函数：

$$J = \frac{1}{2m} \sum(\hat{y} - y)^2 + \frac{\lambda}{2m} \sum w_j^2$$

- λ=0 就是普通线性回归
- λ越大，权重越小，模型越简单

梯度更新：

$$\frac{\partial J}{\partial w} = \frac{1}{m} X^T(\hat{y} - y) + \frac{\lambda}{m} w$$

注意：偏置b不参与正则化！这个一开始我搞错了，把b也正则化了，结果效果很差...后来问了一下AI才知道b不应该正则化，因为b只是偏移量，正则化它没意义

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## 1. 加载数据

In [ ]:
"""
加载加州房价数据集并做预处理

运作流程：
    1. 从sklearn下载加州房价数据集
    2. 拿到特征X、目标y和特征名称
    3. 划分训练集和测试集（8:2）
    4. 对特征做标准化

重要变量：
    - X_train_scaled: 标准化后的训练特征
    - X_test_scaled: 标准化后的测试特征
    - y_train, y_test: 训练和测试目标值
    - feature_names: 特征名称列表

依赖关系：
    - 依赖sklearn的fetch_california_housing、train_test_split、StandardScaler
"""
housing = fetch_california_housing()
X = housing.data
y = housing.target
feature_names = housing.feature_names

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"训练集: {X_train_scaled.shape}")
print(f"测试集: {X_test_scaled.shape}")

## 2. 写岭回归类

跟普通线性回归差不多，就是梯度多了一项 (λ/m)*w

In [ ]:
class MyRidgeRegression:
    """
    岭回归，就是线性回归+L2正则化
    
    属性：
        w: 权重向量
        b: 偏置（不参与正则化！）
        lr: 学习率
        lam: 正则化系数lambda
        n_iters: 迭代次数
        losses: 记录损失
    """
    
    def __init__(self, learning_rate=0.01, lam=1.0, n_iters=1000):
        self.w = None
        self.b = 0
        self.lr = learning_rate
        self.lam = lam
        self.n_iters = n_iters
        self.losses = []
    
    def predict(self, X):
        """
        用训练好的权重和偏置来做预测（跟普通线性回归一模一样）
        
        运作流程：
            1. 把输入特征X和权重w做矩阵乘法
            2. 加上偏置b得到预测值
        
        重要变量：
            - X: 输入特征矩阵，形状(m, n)
            - self.w: 权重向量，形状(n,)
            - self.b: 偏置标量
            - 返回值: 预测值y_hat，形状(m,)
        
        依赖关系：
            - 需要self.w和self.b已经被训练好
            - 依赖numpy的np.dot矩阵乘法
        """
        return np.dot(X, self.w) + self.b
    
    def compute_loss(self, y_hat, y):
        """
        计算岭回归的损失，比普通线性回归多了一个正则化项
        
        运作流程：
            1. 算MSE部分：预测值和真实值的均方误差，跟普通线性回归一样
            2. 算正则化部分：所有权重平方和乘以lambda/(2m)
            3. 把两部分加起来就是总损失
        
        重要变量：
            - y_hat: 预测值，形状(m,)
            - y: 真实值，形状(m,)
            - m: 样本数量
            - mse_part: MSE损失部分，衡量预测准不准
            - reg_part: 正则化损失部分，惩罚太大的权重
            - self.lam: 正则化系数，越大惩罚越重
            - 返回值: 标量总损失
        
        依赖关系：
            - 依赖self.w（计算正则化项）和self.lam
            - 依赖numpy的求和和幂运算
            - 被fit方法调用
        """
        m = len(y)
        mse_part = np.sum((y_hat - y) ** 2) / (2 * m)
        reg_part = (self.lam / (2 * m)) * np.sum(self.w ** 2)
        return mse_part + reg_part
    
    def fit(self, X, y):
        """
        用梯度下降法训练岭回归模型
        
        运作流程：
            1. 初始化权重w为全0向量，偏置b为0
            2. 循环n_iters次迭代：
               a. 调用predict算预测值y_hat
               b. 调用compute_loss算损失（包含MSE+正则化）
               c. 算梯度：dw比普通线性回归多了一项(lambda/m)*w，这就是正则化的效果
                  db不变，因为b不参与正则化！
               d. 沿梯度反方向更新w和b
            3. 返回训练好的self
        
        重要变量：
            - X: 训练特征矩阵，形状(m, n)
            - y: 训练目标向量，形状(m,)
            - m, n: 样本数和特征数
            - diff: 预测值和真实值的差，形状(m,)
            - dw: w的梯度 = 普通梯度 + (lambda/m)*w，多了正则化项
            - db: b的梯度，和普通线性回归一样，不加正则化
            - self.lam: 正则化系数lambda
            - self.losses: 记录每次迭代的损失
        
        依赖关系：
            - 调用self.predict()算预测值
            - 调用self.compute_loss()计算损失
            - 依赖numpy的dot和sum运算
            - 数据需要先标准化
        """
        m, n = X.shape
        self.w = np.zeros(n)
        self.b = 0
        self.losses = []
        
        for i in range(self.n_iters):
            y_hat = self.predict(X)
            loss = self.compute_loss(y_hat, y)
            self.losses.append(loss)
            
            diff = y_hat - y
            dw = (1 / m) * np.dot(X.T, diff) + (self.lam / m) * self.w
            db = (1 / m) * np.sum(diff)
            
            self.w = self.w - self.lr * dw
            self.b = self.b - self.lr * db
        
        return self

## 3. 不同lambda对比

试试不同的正则化系数，看效果有什么区别

In [ ]:
"""
用不同的lambda值训练岭回归模型，对比效果

运作流程：
    1. 设定5个不同的lambda值：0, 0.1, 1, 10, 100
    2. 对每个lambda，创建一个岭回归模型并训练
    3. 记录每个模型的训练损失和权重范数（权重大小）

重要变量：
    - lambdas: lambda值列表
    - models: 训练好的模型列表
    - w_norm: 权重向量的L2范数，衡量权重大小

依赖关系：
    - 依赖前面定义的MyRidgeRegression类
    - 依赖标准化后的训练数据
"""
lambdas = [0, 0.1, 1.0, 10.0, 100.0]
# lambda=0的时候就是普通线性回归，可以对比一下
models = []

for lam in lambdas:
    model = MyRidgeRegression(learning_rate=0.01, lam=lam, n_iters=1000)
    model.fit(X_train_scaled, y_train)
    models.append(model)
    w_norm = np.sqrt(np.sum(model.w ** 2))
    print(f"lambda={lam:6.1f} | 训练损失: {model.losses[-1]:.4f} | 权重范数: {w_norm:.4f}")

## 4. 损失曲线

In [ ]:
"""
画不同lambda下的训练损失曲线

运作流程：
    1. 创建一张图
    2. 对每个lambda对应的模型，画出它的损失随迭代次数的变化
    3. 加标签、图例、网格

重要变量：
    - models[i].losses: 第i个模型每次迭代的损失值
    - lambdas: lambda值列表，用来做图例

依赖关系：
    - 依赖前面训练好的models列表
    - 依赖matplotlib画图
"""
# 可以看到lambda越大收敛越慢，因为正则化项在拉住权重不让它变
plt.figure(figsize=(10, 6))
for i, lam in enumerate(lambdas):
    plt.plot(range(len(models[i].losses)), models[i].losses, label=f'λ={lam}')
plt.xlabel('迭代次数')
plt.ylabel('损失')
plt.title('不同lambda下的训练损失')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 5. 评估

In [ ]:
"""
在测试集上评估不同lambda模型的性能

运作流程：
    1. 对每个lambda对应的模型，在测试集上做预测
    2. 计算MSE、RMSE、R²和权重范数
    3. 打印对比表格

重要变量：
    - y_pred: 每个模型的测试集预测值
    - mse/rmse/r2: 评估指标
    - w_norm: 权重范数，看正则化有没有让权重变小

依赖关系：
    - 依赖前面训练好的models列表
    - 依赖标准化后的X_test_scaled和y_test
"""
print(f"{'lambda':>8} | {'MSE':>8} | {'RMSE':>8} | {'R²':>8} | {'权重范数':>8}")
print("-" * 55)

for i, lam in enumerate(lambdas):
    y_pred = models[i].predict(X_test_scaled)
    mse = np.mean((y_pred - y_test) ** 2)
    rmse = np.sqrt(mse)
    ss_res = np.sum((y_test - y_pred) ** 2)
    ss_tot = np.sum((y_test - np.mean(y_test)) ** 2)
    r2 = 1 - ss_res / ss_tot
    w_norm = np.sqrt(np.sum(models[i].w ** 2))
    print(f"{lam:8.1f} | {mse:8.4f} | {rmse:8.4f} | {r2:8.4f} | {w_norm:8.4f}")

## 6. 权重对比

In [ ]:
"""
画不同lambda下各特征权重的大小对比

运作流程：
    1. 创建一行子图，每个lambda一个
    2. 在每个子图里画柱状图，展示各特征的权重值
    3. 共享y轴方便对比

重要变量：
    - models[i].w: 第i个模型的权重向量
    - feature_names: 特征名称，作为x轴标签
    - lambdas: lambda值列表，作为子图标题

依赖关系：
    - 依赖前面训练好的models列表
    - 依赖feature_names（加载数据时获得）
    - 依赖matplotlib画图
"""
# lambda越大，权重越趋近0，模型越保守
# sharey=True让5个子图共用同一个y轴刻度，方便对比
fig, axes = plt.subplots(1, len(lambdas), figsize=(20, 5), sharey=True)
for i, lam in enumerate(lambdas):
    axes[i].bar(feature_names, models[i].w)
    axes[i].set_title(f'λ = {lam}')
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].grid(True, alpha=0.3)
plt.suptitle('不同lambda下的特征权重')
plt.tight_layout()
plt.show()

## 7. 最优模型可视化

In [ ]:
"""
用lambda=1.0训练最优模型并可视化

运作流程：
    1. 用lambda=1.0重新训练一个岭回归模型
    2. 在测试集上做预测
    3. 画真实值vs预测值的散点图

重要变量：
    - best_model: lambda=1.0的岭回归模型
    - y_pred: 测试集预测值

依赖关系：
    - 依赖MyRidgeRegression类
    - 依赖标准化后的测试数据
"""
best_model = MyRidgeRegression(learning_rate=0.01, lam=1.0, n_iters=1000)
best_model.fit(X_train_scaled, y_train)
y_pred = best_model.predict(X_test_scaled)

plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred, alpha=0.3, s=10)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=2)
plt.xlabel('真实值')
plt.ylabel('预测值')
plt.title('岭回归(λ=1.0)')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
"""
画残差分布图，看模型预测有没有系统性偏差

运作流程：
    1. 算残差 = 真实值 - 预测值
    2. 以预测值为x轴、残差为y轴画散点图
    3. 画一条y=0的红色虚线作为参考线

重要变量：
    - residuals: 残差向量，真实值减预测值
    - y_pred: 测试集预测值（上一个cell算好的）

依赖关系：
    - 依赖上一个cell的best_model和y_pred
    - 依赖y_test
    - 依赖matplotlib画图
"""
residuals = y_test - y_pred
plt.figure(figsize=(8, 5))
plt.scatter(y_pred, residuals, alpha=0.3, s=10)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('预测值')
plt.ylabel('残差')
plt.title('残差分布')
plt.grid(True, alpha=0.3)
plt.show()

## 8. lambda和性能的关系

画一个图看看lambda怎么影响训练误差和测试误差

In [ ]:
"""
探索lambda对模型性能的影响

运作流程：
    1. 在0.001到1000之间取50个lambda值（对数均匀分布）
    2. 对每个lambda训练一个岭回归模型
    3. 记录训练MSE、测试MSE和权重范数
    4. 画两张图：左图看lambda和误差的关系，右图看lambda和权重范数的关系

重要变量：
    - lambda_range: 50个lambda值，从0.001到1000
    - train_errors: 每个lambda对应的训练MSE
    - test_errors: 每个lambda对应的测试MSE
    - w_norms: 每个lambda对应的权重L2范数

依赖关系：
    - 依赖MyRidgeRegression类
    - 依赖标准化后的训练和测试数据
    - 依赖numpy和matplotlib
"""
lambda_range = np.logspace(-3, 3, 50)
# logspace是对数均匀采样，一开始不知道这个函数，问AI才知道有这种写法
# 不然就得手动写 [0.001, 0.01, 0.1, 1, 10, 100] 这样太少了
train_errors = []
test_errors = []
w_norms = []

for lam in lambda_range:
    model = MyRidgeRegression(learning_rate=0.01, lam=lam, n_iters=1000)
    model.fit(X_train_scaled, y_train)
    
    y_train_pred = model.predict(X_train_scaled)
    y_test_pred = model.predict(X_test_scaled)
    
    train_errors.append(np.mean((y_train_pred - y_train) ** 2))
    test_errors.append(np.mean((y_test_pred - y_test) ** 2))
    w_norms.append(np.sqrt(np.sum(model.w ** 2)))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# semilogx就是把x轴变成对数刻度，这样lambda从0.001到1000都能看得清楚
ax1.semilogx(lambda_range, train_errors, 'b-', label='训练MSE')
ax1.semilogx(lambda_range, test_errors, 'r-', label='测试MSE')
ax1.set_xlabel('lambda')
ax1.set_ylabel('MSE')
ax1.set_title('lambda与MSE的关系')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.semilogx(lambda_range, w_norms, 'g-')
ax2.set_xlabel('lambda')
ax2.set_ylabel('权重范数')
ax2.set_title('lambda与权重范数的关系')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()